# Notebook 4 — Final temporal evaluation and interpretation
**One-time official holdout · frozen M0–M4 · 15 URL + 10 HTML features.**

Start with **Cell 2 `RUN_MODE = "SMOKE"`**, CPU/no accelerator, and Run All. SMOKE uses fabricated data and small fabricated models only; its numbers have no scientific meaning. No real inputs are needed. After SMOKE PASS, attach saved FULL NB2 outputs (test feature Parquet and feature JSON; NB2 manifest recommended) and saved FULL NB3 outputs (five joblib models, frozen config, temporal split and manifest). Change mode to FULL, restart the session and Run All **once**. Save the executed Kaggle version with outputs. Do not rerun NB3 to recover files unless the saved models are genuinely lost.

Paths can be flat or nested. If automatic discovery is ambiguous, set `NB2_INPUT` and `NB3_INPUT` to the specific attached folders. If models exist only in the trusted `NB03_FULL_ALL_OUTPUTS.zip`, set `NB3_ZIP` to its exact mounted path. No raw HTML, training Parquet, crawling, DNS or WHOIS is used. No real model is fitted, tuned, recalibrated or selected here. Test results must never inform later tuning. If a technical bug requires repeating FULL, document it in `REPEAT_EXPOSURE_NOTE` and retain the prior run.

**Predeclared methods:** fixed saved `.predict()` labels; class-1 probabilities; primary measure sklearn Average Precision (AP), not trapezoidal PR area. P@R=.90 uses the highest score threshold reaching ≥.90 recall, including the entire tied-score group; this descriptive threshold never changes model predictions. Exactly two McNemar comparisons (LR and RF URL vs hybrid), unadjusted α=.05, exact test below 25 discordant rows and continuity-corrected asymptotic otherwise. Paired row bootstrap: 500 successful FULL resamples, seed 42, 95% percentile intervals for F1/MCC/AP and hybrid-minus-URL differences. Permutation importance: M4 AP, five repeats, seed 42, fixed random subset of at most 10,000 test rows. Neither procedure refits models. Row-bootstrap intervals assume independent rows and may understate uncertainty when related pages cluster.

**Boundary caveat:** official train and test contain benign samples on the same calendar day, 2025-09-08; the boundary is not globally strictly temporal. Validation APs are historical context only. No verified published same-test comparator was supplied; F30 therefore shows this project's own measured AP with the missing comparator stated explicitly. The optional 5% base-rate experiment is OFF.

Outputs: `/kaggle/working/nb04/smoke` or `/kaggle/working/nb04/full`, plus a separate self-excluding ZIP. F19–F30 are 300-DPI PNGs. No raw URLs are exported. This notebook has **exactly 15 physical cells**; headings for code cells are comments.


In [ ]:
# CELL 2 — Settings, dependencies, discovery, verified loading; synthetic-only SMOKE
RUN_MODE = "SMOKE"
NB2_INPUT = ""  # Optional exact relevant Kaggle input folder; blank = bounded auto-discovery.
NB3_INPUT = ""
NB3_ZIP = ""    # Optional explicit trusted NB03_FULL_ALL_OUTPUTS.zip path (no arbitrary ZIPs).
INPUT_ROOT = "/kaggle/input"
OUTPUT_ROOT = "/kaggle/working/nb04"
MAKE_ZIP = True
SAVE_PDF = False
REPEAT_EXPOSURE_NOTE = ""  # Required if correcting a bug after any previous FULL test exposure.

import os, sys, json, hashlib, time, shutil, platform, zipfile, warnings, gc, stat
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import joblib, sklearn
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, average_precision_score, confusion_matrix,
    precision_recall_curve, roc_curve)
from sklearn.inspection import permutation_importance

def require(ok, message):
    if not bool(ok): raise RuntimeError("STOP: " + message)
require(RUN_MODE in {"SMOKE", "FULL"}, "Choose SMOKE or FULL.")
SEED = 42
BOOTSTRAP_B = 500 if RUN_MODE == "FULL" else 20
PERMUTATION_N = 10000 if RUN_MODE == "FULL" else 120
PERMUTATION_REPEATS = 5
STARTED = time.perf_counter()
RUN_ID = datetime.now(timezone.utc).isoformat()
OUT = Path(OUTPUT_ROOT) / RUN_MODE.lower()
for folder in ["predictions", "metrics", "tables", "manifests", "reports", "figures"]:
    (OUT / folder).mkdir(parents=True, exist_ok=True)

def safe_json(v):
    if isinstance(v, np.generic): return safe_json(v.item())
    if isinstance(v, dict): return {str(k): safe_json(x) for k, x in v.items()}
    if isinstance(v, (list, tuple, np.ndarray)): return [safe_json(x) for x in v]
    if isinstance(v, float) and not np.isfinite(v): return str(v)
    if isinstance(v, Path): return str(v)
    return v

def write_json(path, value):
    path = Path(path); tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(safe_json(value), indent=2, allow_nan=False), encoding="utf-8")
    tmp.replace(path)

MANIFEST_PATH = OUT / "manifests/manifest_nb04.json"
prior = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else {}
# Archive previous status and invalidate success BEFORE imports, discovery or loading can fail.
if prior:
    archive = OUT.parent / "run_history" / (RUN_MODE.lower() + "_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f") + ".json")
    archive.parent.mkdir(exist_ok=True); write_json(archive, prior)
status = {"run_id": RUN_ID, "mode": RUN_MODE, "official_test_loaded": False,
          "official_test_evaluated": False, "test_prediction_started": False,
          "full_complete": False, "smoke_complete": False,
          "prior_test_exposure": bool(prior.get("prior_test_exposure") or prior.get("test_prediction_started") or prior.get("official_test_evaluated")),
          "blockers": ["Run incomplete; reach Cell 15."], "repeat_exposure_note": REPEAT_EXPOSURE_NOTE}
write_json(MANIFEST_PATH, status)
if RUN_MODE == "FULL" and status["prior_test_exposure"]:
    require(bool(REPEAT_EXPOSURE_NOTE.strip()),
        "A prior FULL exposure is recorded. Preserve prior outputs and document a genuine technical correction in REPEAT_EXPOSURE_NOTE; do not retune.")
try:
    import pyarrow, openpyxl, psutil, statsmodels
    from statsmodels.stats.contingency_tables import mcnemar
except ImportError as exc:
    raise RuntimeError("Install missing packages (statsmodels==0.14.5, pyarrow, openpyxl, psutil), restart session, Run All. " + str(exc)) from exc
packages = {p: version(p) for p in ["numpy", "pandas", "scikit-learn", "scipy", "joblib",
    "pyarrow", "matplotlib", "openpyxl", "psutil", "statsmodels"]}
packages["python"] = platform.python_version()
print("Mode:", RUN_MODE, "\nPackages:", json.dumps(packages, indent=2))
# Exact serialization stack; pure data/plot package versions are recorded but not load blockers.
SERIAL_STACK = {"scikit-learn": "1.6.1", "numpy": "2.1.3", "scipy": "1.16.3", "joblib": "1.6.0"}
if RUN_MODE == "FULL":
    require(sys.version_info[:2] == (3, 13),
        "Frozen models were saved with Python 3.13.15. Use a Python 3.13 runtime; pip cannot change Python. Do not retrain or bypass compatibility checks.")
    mismatches = {k: (packages[k], v) for k, v in SERIAL_STACK.items() if packages[k] != v}
    require(not mismatches and packages["statsmodels"] == "0.14.5",
        "Dependency mismatch " + str(mismatches) + ". In a temporary installation cell run: %pip install scikit-learn==1.6.1 numpy==2.1.3 scipy==1.16.3 joblib==1.6.0 statsmodels==0.14.5 ; then delete that cell, restart and Run All. Keep 15 physical cells. No silent installation/loading.")

U = ["url_length","hostname_length","path_length","query_length","num_dots",
     "num_hyphens","num_digits","digit_ratio","num_subdomains","path_depth",
     "num_query_params","has_ip_host","has_punycode","has_at_symbol","url_entropy"]
H = ["anchor_external_ratio","form_external_ratio","form_invalid_action_ratio",
     "script_external_ratio","image_external_ratio","iframe_external_ratio",
     "link_resource_external_ratio","overall_external_resource_ratio",
     "unique_external_registered_domains","dominant_resource_domain_mismatch"]
F = U + H
IDS = ["M0", "M1", "M2", "M3", "M4"]
NAMES = dict(zip(IDS, ["Majority", "LR URL", "LR URL+HTML", "RF URL", "RF URL+HTML"]))
FEATURES = {m: (["dummy_constant"] if m == "M0" else U if m in ["M1", "M3"] else F) for m in IDS}
MODEL_FILES = dict(zip(IDS, ["M0_dummy_majority.joblib", "M1_lr_url.joblib", "M2_lr_hybrid.joblib", "M3_rf_url.joblib", "M4_rf_hybrid.joblib"]))
EXPECTED = {
 "hybrid_features_test.parquet": (18725470, "7d0bcc0835bb1d89a7147dc7b81412b193bbd76ca6692aeb11db49685fbf8e80"),
 "final_feature_set.json": (3877, "3bb24f1e9c1ffc6a02d0a7307e60fa78c68fc0f1dedb3309d0ab025cd97d729a"),
 "M0_dummy_majority.joblib": (784, "8c5f4a083656544b6cb440e4bcbc71e763907cae7c0dd766bbe39b8f890b0b33"),
 "M1_lr_url.joblib": (1914, "63f3a7f06cce2006b774b0cc06a4bc7065558490a7853630a50162150ac8547e"),
 "M2_lr_hybrid.joblib": (2423, "5108d94590a10c53ca9bd1eb5d6f0ecc0b4ba7f913d0b670acc6befca2298885"),
 "M3_rf_url.joblib": (135549608, "fcf0bbce6135be255434d39901c763165e3d45c185a3a6798f8ddce5b8bb685c"),
 "M4_rf_hybrid.joblib": (114006730, "7721e9a9be88c757b8ac99de29bf4fb0a882f6865b5c85f1bb5d5ff072f9620c"),
 "frozen_model_config.json": (10357, "02609b5b0dbfea73548bc74e8228104c2256abad2685663eff438655d9bad057"),
 "temporal_split.json": (1089, "8cf1a806271cd2f1fda9467a2e2b096ea7f3decbf44775ebc6ea5857bec839e7")}
DATASET = {"id":"phreshphish/phreshphish", "version":"1.0.1", "revision":"eabec4b7a66324b79cc8a0ad856d1731dc26fe1a"}
BOUNDARY = "Train and test share benign calendar date 2025-09-08; not globally strictly temporally separated."
inputs, input_records, models, model_paths = {}, {}, {}, {}
figures, tables, figure_rows = [], {}, []
blockers = []
OFFICIAL_EVALUATED = False

# Helpers are also exercised on fabricated files during focused tests.
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(2**20), b""): h.update(chunk)
    return h.hexdigest()
def file_record(path):
    return {"path": str(path), "bytes": Path(path).stat().st_size, "sha256": sha256_file(path)}
def verify_file(path, expected):
    rec = file_record(path)
    require((rec["bytes"], rec["sha256"]) == tuple(expected), f"Size/SHA mismatch: {path}. Restore the exact FULL artifact; never change expected hashes to force PASS.")
    return rec

def discover(base, name):
    base = Path(base)
    hits = {p for p in base.rglob(name) if p.is_file()}
    if not hits:
        # Follow symlinks but prune loops; hard bound for diagnostics and unrelated inputs.
        seen, count = set(), 0
        for current, dirs, files in os.walk(base, followlinks=True):
            resolved = Path(current).resolve()
            if resolved in seen: dirs[:] = []; continue
            seen.add(resolved); count += len(files)
            require(count <= 20000 and len(seen) <= 3000, f"Discovery too broad at {base}; set exact NB2_INPUT/NB3_INPUT.")
            if name in files: hits.add(Path(current) / name)
    return sorted(hits)

def diagnostic(root):
    # Tiny directory inventory only; never open data files.
    shown = 0
    if not Path(root).exists(): print("Input root missing:", root); return
    for cur, dirs, files in os.walk(root, followlinks=False):
        print(cur, "files:", files[:8]); shown += 1
        if shown >= 12: break

def one(base, name):
    hits = discover(base, name)
    if len(hits) != 1: diagnostic(base)
    require(len(hits) == 1, f"Expected exactly one {name} under {base}; got {[str(p) for p in hits]}. Remove duplicate mounts or set exact input folder.")
    return hits[0]

def mounted_roots(root):
    root = Path(root)
    if not root.exists(): return []
    result = []
    for p in sorted(root.iterdir()):
        if not p.is_dir(): continue
        if p.name in {"datasets", "notebooks"}:
            for owner in sorted(p.iterdir()):
                if owner.is_dir(): result.extend(x for x in sorted(owner.iterdir()) if x.is_dir())
        else: result.append(p)
    return result

def resolve_mount(explicit, anchor):
    if explicit:
        p = Path(explicit); require(p.is_dir(), f"Missing folder: {p}"); one(p, anchor); return p
    candidates = []
    # Look only at filenames in bounded attachment trees; no unrelated file contents are read.
    for mount in mounted_roots(INPUT_ROOT):
        scanned = 0; seen = set(); found = False
        for cur, dirs, files in os.walk(mount, followlinks=True):
            real = Path(cur).resolve()
            if real in seen: dirs[:] = []; continue
            seen.add(real); scanned += len(files)
            if anchor in files: found = True; break
            if scanned > 2000 or len(seen) > 500: break
        if found: candidates.append(mount)
    if len(candidates) != 1: diagnostic(INPUT_ROOT)
    require(len(candidates) == 1, f"Need one attachment containing {anchor}, found {candidates}. Set NB2_INPUT/NB3_INPUT to its precise folder.")
    return candidates[0]

def extract_trusted_nb3(path):
    path = Path(path)
    require(path.name == "NB03_FULL_ALL_OUTPUTS.zip", "ZIP fallback accepts only the explicitly supplied NB03_FULL_ALL_OUTPUTS.zip.")
    dest = OUT / "_verified_nb03_zip"
    allowed = set(MODEL_FILES.values()) | {"manifest_nb03.json", "frozen_model_config.json", "temporal_split.json"}
    with zipfile.ZipFile(path) as z:
        chosen = {}
        for entry in z.infolist():
            parts = PurePosixPath(entry.filename)
            require(not parts.is_absolute() and ".." not in parts.parts and "\\" not in entry.filename,
                    "Unsafe ZIP path rejected.")
            require(not stat.S_ISLNK(entry.external_attr >> 16), "ZIP symlink rejected.")
            if parts.name in allowed and not entry.is_dir():
                require(parts.name not in chosen, "Duplicate expected ZIP member: " + parts.name)
                chosen[parts.name] = entry
        require(set(chosen) == allowed, "ZIP missing frozen artifacts.")
        for name, entry in chosen.items():
            limit = EXPECTED[name][0] if name in EXPECTED else 5 * 2**20
            require(entry.file_size <= limit, "Unexpected oversized ZIP member: " + name)
        require(shutil.disk_usage(OUT).free > sum(x.file_size for x in chosen.values()) + 512*2**20, "Insufficient extraction disk.")
        dest.mkdir(exist_ok=True)
        for name, entry in chosen.items():
            target = dest / name; tmp = target.with_suffix(target.suffix + ".tmp")
            with z.open(entry) as src, tmp.open("wb") as dst: shutil.copyfileobj(src, dst)
            if name in EXPECTED: verify_file(tmp, EXPECTED[name])
            tmp.replace(target)
    return dest

if RUN_MODE == "FULL":
    require(psutil.virtual_memory().available > 2*2**30 and shutil.disk_usage(OUT).free > 2*2**30,
            "Need at least 2 GiB available RAM and output disk; free resources/restart.")
    nb2_mount = resolve_mount(NB2_INPUT, "hybrid_features_test.parquet")
    nb3_mount = extract_trusted_nb3(NB3_ZIP) if NB3_ZIP else resolve_mount(NB3_INPUT, "manifest_nb03.json")
    for name in EXPECTED:
        inputs[name] = one(nb2_mount if name in ["hybrid_features_test.parquet", "final_feature_set.json"] else nb3_mount, name)
    inputs["manifest_nb03.json"] = one(nb3_mount, "manifest_nb03.json")
    optional = discover(nb2_mount, "manifest_nb02.json")
    require(len(optional) <= 1, "Duplicate NB2 manifests.")
    if optional: inputs["manifest_nb02.json"] = optional[0]
    for name, path in inputs.items():
        print(name, "->", path)
        input_records[name] = verify_file(path, EXPECTED[name]) if name in EXPECTED else file_record(path)
    # Every frozen file hash above is checked BEFORE reading the test or deserializing a model.
    nb3 = json.loads(inputs["manifest_nb03.json"].read_text())
    config = json.loads(inputs["frozen_model_config.json"].read_text())
    fs = json.loads(inputs["final_feature_set.json"].read_text())
    temporal = json.loads(inputs["temporal_split.json"].read_text())
    require(nb3.get("mode") == "FULL" and nb3.get("ready_for_nb04") is True and nb3.get("blockers") == [] and nb3.get("config_frozen") is True,
            "NB3 is not a ready, frozen, blocker-free FULL run.")
    require(all(nb3.get(k) is False and config.get(k) is False for k in ["official_test_loaded", "official_test_evaluated"]), "NB3 embargo provenance failed.")
    CONFIG_HASH = input_records["frozen_model_config.json"]["sha256"]
    require(nb3.get("config_sha256") == CONFIG_HASH, "NB3 config hash differs.")
    require(config["mode"] == "FULL" and config["config_frozen"] is True and config["refit_rows"] == 498255 and config["threshold"] == .5 and config["positive_label"] == 1, "Frozen config contract failed.")
    require(config["dataset"] == DATASET and nb3["dataset"] == DATASET, "Dataset revision mismatch.")
    require(config["packages"] == nb3["packages"], "NB3 package manifests disagree.")
    for k, v in SERIAL_STACK.items(): require(config["packages"][k] == v, "Saved stack mismatch: " + k)
    for name in list(MODEL_FILES.values()) + ["frozen_model_config.json", "temporal_split.json"]:
        records = [r for rel, r in nb3["artifacts"].items() if Path(rel).name == name]
        require(len(records) == 1 and records[0]["sha256"] == input_records[name]["sha256"] and records[0]["bytes"] == input_records[name]["bytes"], "NB3 artifact record differs: " + name)
    require(fs["mode"] == "FULL" and fs["official_test_evaluated"] is False and fs["internal_cutoff_inclusive"][:10] == "2025-07-09", "Feature selection provenance mismatch.")
    if "manifest_nb02.json" in inputs:
        nb2 = json.loads(inputs["manifest_nb02.json"].read_text())
        require(nb2.get("mode") == "FULL" and nb2.get("blockers") == [] and nb2.get("official_test_evaluated") is False and nb2.get("ready_for_nb03") is True, "NB2 provenance failed.")
        # NB2 may nest artifacts; locate complete matching hash in the relevant record only.
        def artifact_hits(obj, name, context=""):
            if isinstance(obj, dict):
                if name in context or any(name in str(v) for v in obj.values() if isinstance(v, str)): yield obj
                for k, v in obj.items(): yield from artifact_hits(v, name, str(k))
            elif isinstance(obj, list):
                for v in obj: yield from artifact_hits(v, name, context)
        for name in ["hybrid_features_test.parquet", "final_feature_set.json"]:
            require(any(input_records[name]["sha256"] in json.dumps(h) for h in artifact_hits(nb2["artifacts"], name)), "NB2 missing artifact hash: " + name)
    else: print("NB2 manifest not supplied; exact handoff-pinned sizes/hashes establish artifact identity.")
    for mid in IDS:
        path = inputs[MODEL_FILES[mid]]
        with warnings.catch_warnings():
            warnings.simplefilter("error", sklearn.exceptions.InconsistentVersionWarning)
            models[mid] = joblib.load(path)
        model_paths[mid] = path
    df = pd.read_parquet(inputs["hybrid_features_test.parquet"], columns=["sha256", "url", "label", "date"] + F).reset_index(drop=True)
    status["official_test_loaded"] = True; write_json(MANIFEST_PATH, status)
else:
    print("SMOKE: fabricated data/models only; no Kaggle input discovery or official-test reads.")
    fixture = OUT / "_synthetic_fixtures"; fixture.mkdir(exist_ok=True)
    rng = np.random.default_rng(SEED)
    def fabricated(n, prefix):
        x = pd.DataFrame(rng.uniform(0, 1, (n, 25)), columns=F)
        x["unique_external_registered_domains"] = rng.integers(0, 8, n)
        x["dominant_resource_domain_mismatch"] = rng.integers(0, 2, n)
        y_fake = (2*x.url_length + 2*x.anchor_external_ratio + rng.normal(0, .5, n) > 2.45).astype(int)
        x.loc[rng.choice(n, max(1,n//20), replace=False), "path_length"] = np.nan
        x["label"] = y_fake
        x["sha256"] = [hashlib.sha256(f"{prefix}-{i}".encode()).hexdigest() for i in range(n)]
        x["url"] = "https://synthetic.invalid/"
        x["date"] = pd.Timestamp("2025-10-01", tz="UTC")
        return x
    train_fake, df = fabricated(360, "train-fake"), fabricated(240, "evaluation-fake")
    # Synthetic fitting is confined to this branch and happens before any final metrics.
    require(train_fake.label.value_counts().idxmax() == 0, "Synthetic majority must be benign.")
    fs = {"url_only_features": U, "retained_h_features": H, "combined_features": F}
    config = {"mode":"SMOKE", "models":{}, "refit_rows":len(train_fake), "run_id":RUN_ID,
              "refit_scope":"fabricated synthetic training only", "config_frozen":True}
    for mid in IDS:
        if mid == "M0": model = Pipeline([("model", DummyClassifier(strategy="most_frequent", random_state=SEED))])
        elif mid in ["M1", "M2"]:
            model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
                ("model", LogisticRegression(C=10., penalty="l2", solver="lbfgs", max_iter=5000, random_state=SEED))])
        else:
            model = Pipeline([("imputer", SimpleImputer(strategy="median")),
                ("model", RandomForestClassifier(n_estimators=12, max_depth=4, random_state=SEED, n_jobs=-1))])
        x = pd.DataFrame({"dummy_constant":np.ones(len(train_fake))}) if mid == "M0" else train_fake[FEATURES[mid]]
        model.fit(x, train_fake.label)
        config["models"][mid] = {"features":FEATURES[mid], "estimator_class":type(model.named_steps["model"]).__name__,
            "estimator_parameters":safe_json(model.named_steps["model"].get_params(deep=False)),
            "preprocessing":{k:safe_json(v.get_params(deep=False)) for k,v in model.steps[:-1]}}
        models[mid] = model
    config_path = fixture / "synthetic_config.json"; write_json(config_path, config); CONFIG_HASH = sha256_file(config_path)
    for mid, model in models.items():
        model.nb03_provenance_ = {"model_id":mid, "mode":"SMOKE", "run_id":RUN_ID, "fit_rows":len(train_fake),
            "fit_scope":config["refit_scope"], "features":FEATURES[mid], "config_sha256":CONFIG_HASH,
            "official_test_loaded":False, "official_test_evaluated":False}
        p = fixture / MODEL_FILES[mid]; joblib.dump(model,p); model_paths[mid] = p
        rec = file_record(p); verify_file(p,(rec["bytes"],rec["sha256"])); models[mid] = joblib.load(p)
    del train_fake, model, x
    nb3 = {"mode":"SMOKE", "synthetic":True}
MODEL_RECORDS = {m:file_record(p) for m,p in model_paths.items()}
MODEL_STATE_HASHES = {m:joblib.hash(model) for m,model in models.items()}
plt.rcParams.update({"figure.dpi":110,"savefig.dpi":300,"font.size":10,"axes.spines.top":False,
                     "axes.spines.right":False,"axes.grid":False,"figure.facecolor":"white"})
COLORS = ["#777777", "#0072B2", "#56B4E9", "#D55E00", "#009E73"]
SCOPE = "Official test" if RUN_MODE == "FULL" else "SYNTHETIC SMOKE — NOT THESIS RESULTS"
def savefig(fig, name, caption):
    fig.suptitle(SCOPE, fontsize=10, color="#555555")
    fig.tight_layout(rect=[0,.055,1,.94])
    fig.text(.01,.01,caption,fontsize=8,va="bottom",wrap=True)
    rel = "figures/" + name; fig.savefig(OUT/rel,dpi=300,facecolor="white")
    if SAVE_PDF: fig.savefig((OUT/rel).with_suffix(".pdf"),facecolor="white")
    plt.close(fig); figures.append(rel)
    figure_rows.append({"figure":name.split("_")[0],"file":rel,"caption":caption,"scope":SCOPE,"run_utc":RUN_ID})
def save_table(rel, frame):
    frame.to_csv(OUT/rel,index=False); tables[rel] = frame
print("Preflight loaded", len(models), "models; all genuine-model loads require pinned checksums.")


In [ ]:
# CELL 3 — Test identity/schema and exact frozen model contracts
require(df.columns.is_unique and set(["sha256","url","label","date"]+F).issubset(df.columns), "Missing/duplicate schema columns.")
require(fs["url_only_features"] == U and fs["retained_h_features"] == H and fs["combined_features"] == F, "Frozen feature order differs.")
require(len(F) == 25 and len(set(F)) == 25, "Invalid feature allowlist.")
require(not df[["sha256","url","label","date"]].isna().any().any(), "Missing required metadata/labels.")
require(df.sha256.is_unique and df.sha256.astype(str).str.fullmatch(r"[0-9a-f]{64}").all(), "Duplicate/invalid SHA identities.")
require(set(df.label.unique()) == {0,1}, "Binary labels must be 0 and 1.")
require(all(pd.api.types.is_numeric_dtype(df[c]) for c in F), "Features must be numeric.")
require(not np.isinf(df[F].to_numpy(dtype=float)).any(), "Infinite features rejected; NaNs use saved imputation.")
for c in H:
    v = df[c].dropna()
    require(((v >= 0) & (v % 1 == 0)).all() if c == "unique_external_registered_domains" else v.between(0,1).all(), "Invalid HTML feature range: " + c)
dates = pd.to_datetime(df.date, utc=True, errors="raise")
y = df.label.to_numpy(dtype=np.int8)
n = len(df)
if RUN_MODE == "FULL":
    require(n == 168060 and int((y==0).sum()) == 91260 and int(y.sum()) == 76800, "Official test counts differ.")
    require(dates.min().strftime("%Y-%m-%d") == "2025-09-08" and dates.max().strftime("%Y-%m-%d") == "2025-12-16", "Official test date endpoints differ.")
    require(inputs["hybrid_features_test.parquet"].name == "hybrid_features_test.parquet", "Training input forbidden.")
for mid, model in models.items():
    spec = config["models"][mid]
    require(isinstance(model,Pipeline) and list(model.feature_names_in_) == FEATURES[mid] == spec["features"], mid + " feature order mismatch.")
    expected_steps = ["model"] if mid == "M0" else ["imputer","scaler","model"] if mid in ["M1","M2"] else ["imputer","model"]
    require(list(model.named_steps) == expected_steps, mid + " pipeline steps differ.")
    estimator = model.named_steps["model"]
    require(type(estimator).__name__ == spec["estimator_class"], mid + " estimator type differs.")
    require(safe_json(estimator.get_params(deep=False)) == spec["estimator_parameters"], mid + " estimator parameters differ from config.")
    for key, step in model.steps[:-1]:
        require(safe_json(step.get_params(deep=False)) == spec["preprocessing"][key], mid + " preprocessing differs.")
    if mid != "M0": require(model.named_steps["imputer"].strategy == "median" and len(model.named_steps["imputer"].statistics_) == len(FEATURES[mid]), "Saved imputer contract failed.")
    if mid in ["M1","M2"]: require(estimator.C == 10. and estimator.penalty == "l2", "LR frozen setting mismatch.")
    if mid in ["M3","M4"] and RUN_MODE == "FULL": require(estimator.n_estimators == 300 and estimator.max_depth == 20 and estimator.random_state == 42 and estimator.n_jobs == -1, "RF frozen setting mismatch.")
    provenance = model.nb03_provenance_
    expected_provenance = {"model_id":mid,"mode":RUN_MODE,"run_id":config["run_id"],
        "fit_rows":498255 if RUN_MODE == "FULL" else config["refit_rows"],"fit_scope":config["refit_scope"],
        "features":FEATURES[mid],"config_sha256":CONFIG_HASH,"official_test_loaded":False,"official_test_evaluated":False}
    require(provenance == expected_provenance, mid + " refit provenance mismatch.")
    require(set(model.classes_.tolist()) == {0,1}, mid + " expected classes 0 and 1.")
require(models["M0"].named_steps["model"].strategy == "most_frequent" and int(np.argmax(models["M0"].named_steps["model"].class_prior_)) == 0, "M0 saved majority must be benign.")
TEST_PREVALENCE = float(y.mean())
# Subset indices fixed before seeing any scores or final performance. No stratification/rebalancing.
perm_indices = np.sort(np.random.default_rng(SEED).choice(n,size=min(PERMUTATION_N,n),replace=False))
PERM_INFO = {"selection":"uniform random rows without replacement, sorted indices; fixed before scores", "n":len(perm_indices),
    "seed":SEED,"n_repeats":PERMUTATION_REPEATS,"scoring":"average_precision",
    "indices_sha256":hashlib.sha256(perm_indices.astype("<i8").tobytes()).hexdigest(),
    "sample_prevalence":float(y[perm_indices].mean())}
require(len(np.unique(y[perm_indices])) == 2, "Predeclared permutation sample has one class; report blocker, do not choose a better sample.")
ROW_HASH = hashlib.sha256("\n".join(df.sha256).encode()).hexdigest()
print(f"Schema PASS: n={n:,}; benign={(y==0).sum():,}; phishing={y.sum():,}; prevalence={TEST_PREVALENCE:.6f}")
print(BOUNDARY)


In [ ]:
# CELL 4 — Frozen predictions and timings, identical aligned rows
predictions, scores, timing_rows = {}, {}, []
status["test_prediction_started"] = RUN_MODE == "FULL"; write_json(MANIFEST_PATH,status)
def model_x(mid):
    return pd.DataFrame({"dummy_constant":np.ones(n,dtype=float)}) if mid == "M0" else df[FEATURES[mid]]
for mid in IDS:
    model = models[mid]; x = model_x(mid)
    positive_column = int(np.flatnonzero(model.classes_ == 1)[0])
    # Same warmup: first min(128,n) rows; one timed full call per API, no fitting.
    model.predict_proba(x.iloc[:min(128,n)]); model.predict(x.iloc[:min(128,n)])
    rss_before = psutil.Process().memory_info().rss
    start = time.perf_counter(); proba = model.predict_proba(x); proba_s = time.perf_counter()-start
    start = time.perf_counter(); pred = model.predict(x); predict_s = time.perf_counter()-start
    score = proba[:,positive_column]
    require(len(pred)==n and len(score)==n and np.isfinite(proba).all() and np.allclose(proba.sum(axis=1),1), mid+" malformed predictions.")
    require(set(np.unique(pred)).issubset({0,1}) and ((score>=0)&(score<=1)).all(), mid+" invalid values.")
    require(np.array_equal(pred,model.classes_[np.argmax(proba,axis=1)]),mid+" predict/probability rule mismatch.")
    predictions[mid] = pred.astype(np.int8); scores[mid] = score.copy()
    timing_rows.append({"model_id":mid,"n":n,"predict_proba_seconds":proba_s,"predict_seconds":predict_s,
        "combined_seconds":proba_s+predict_s,"combined_ms_per_1000":(proba_s+predict_s)*1e6/n,
        "predict_proba_ms_per_1000":proba_s*1e6/n,"predict_ms_per_1000":predict_s*1e6/n,
        "rss_before_MiB":rss_before/2**20,"rss_after_MiB":psutil.Process().memory_info().rss/2**20,
        "timing_scheme":"128-row warmup, one full predict_proba then one full predict; feature-frame preparation excluded; RSS snapshots not peak"})
    print(mid,NAMES[mid],f"inference {proba_s+predict_s:.3f} s",flush=True)
    del x, proba, pred; gc.collect()
require(np.all(predictions["M0"]==0) and np.all(scores["M0"]==0),"M0 must predict benign with class-1 score 0.")


In [ ]:
# CELL 5 — Full metrics and predeclared descriptive precision at recall .90
# No search for best precision: take the HIGHEST threshold meeting required recall, with all ties included.
def precision_at_recall(y_true, score, target=.90):
    p,r,t = precision_recall_curve(y_true,score)
    eligible = np.flatnonzero(r[:-1] >= target)
    require(len(eligible)>0,"No threshold reaches target recall.")
    ix = int(eligible[-1])
    chosen = score >= t[ix]
    require(np.isclose(recall_score(y_true,chosen),r[ix]) and np.isclose(precision_score(y_true,chosen),p[ix]),"P@R tie handling failed.")
    return {"target_recall":target,"precision_at_recall":float(p[ix]),"achieved_recall":float(r[ix]),
            "descriptive_threshold":float(t[ix]),"selected_rows":int(chosen.sum()),
            "rule":"highest threshold reaching recall >= .90; include entire tied score group; does NOT alter predict()"}
def metric_record(yy, pp, ss):
    tn,fp,fn,tp = confusion_matrix(yy,pp,labels=[0,1]).ravel()
    return {"accuracy":accuracy_score(yy,pp),"precision":precision_score(yy,pp,zero_division=0),
        "recall":recall_score(yy,pp,zero_division=0),"F1":f1_score(yy,pp,zero_division=0),
        "MCC":matthews_corrcoef(yy,pp),"ROC_AUC":roc_auc_score(yy,ss),"AP":average_precision_score(yy,ss),
        "FPR":float(fp/(tn+fp)),"TN":int(tn),"FP":int(fp),"FN":int(fn),"TP":int(tp)}
metric_rows, pr90_rows = [], []
for mid in IDS:
    rec = metric_record(y,predictions[mid],scores[mid]); rec.update(model_id=mid,model=NAMES[mid],n=n,prevalence=TEST_PREVALENCE,scope=SCOPE)
    require(rec["TN"]+rec["FP"]+rec["FN"]+rec["TP"]==n and rec["FN"]+rec["TP"]==int(y.sum()), "Confusion arithmetic failed.")
    require(np.isclose(rec["accuracy"],(rec["TN"]+rec["TP"])/n),"Accuracy identity failed.")
    metric_rows.append(rec); pr90_rows.append({"model_id":mid,**precision_at_recall(y,scores[mid])})
metrics = pd.DataFrame(metric_rows).set_index("model_id",drop=False)
pr90 = pd.DataFrame(pr90_rows)
require(np.isclose(metrics.loc["M0","AP"],TEST_PREVALENCE) and metrics.loc["M0","ROC_AUC"]==.5 and metrics.loc["M0","F1"]==0,"Trivial M0 metrics failed.")
save_table("metrics/final_test_metrics.csv",metrics.reset_index(drop=True))
save_table("metrics/final_confusion_counts.csv",metrics[["model_id","TN","FP","FN","TP","n"]].reset_index(drop=True))
save_table("metrics/precision_at_recall_090.csv",pr90)
OFFICIAL_EVALUATED = RUN_MODE == "FULL"
status["official_test_evaluated"] = OFFICIAL_EVALUATED; write_json(MANIFEST_PATH,status)
print(metrics[["model","accuracy","precision","recall","F1","MCC","ROC_AUC","AP","FPR"]].to_string(index=False))
print("P@R=.90 is descriptive; fixed classification predictions remain unchanged.")


In [ ]:
# CELL 6 — F19–F23 metrics, PR/ROC, confusion matrices, paired deltas
PAIRS = [("M1","M2"),("M3","M4")]
DELTA_METRICS = ["accuracy","precision","recall","F1","MCC","ROC_AUC","AP","FPR"]
delta_rows=[]
for url,hybrid in PAIRS:
    for metric in DELTA_METRICS:
        raw = float(metrics.loc[hybrid,metric]-metrics.loc[url,metric])
        delta_rows.append({"comparison":hybrid+"-"+url,"metric":metric,"hybrid_minus_url":raw,
            "raw_unit":metric+" points (0–1 scale)" if metric!="MCC" else "MCC units",
            "percentage_point_difference":raw*100 if metric!="MCC" else np.nan,
            "desirable_direction":"negative" if metric=="FPR" else "positive"})
deltas=pd.DataFrame(delta_rows); save_table("metrics/hybrid_minus_url_differences.csv",deltas)
caption=f"{SCOPE}; n={n:,}; fixed NB3 models. AP uses average_precision_score; prevalence={TEST_PREVALENCE:.3f}."
fig,ax=plt.subplots(figsize=(9,4.5))
metrics[["AP","F1","MCC"]].plot.bar(ax=ax,color=["#0072B2","#009E73","#D55E00"],rot=0)
ax.set(xlabel="Frozen model",ylabel="Metric value",title="Final discrimination and classification metrics",ylim=(min(0,float(metrics.MCC.min())-.03),1.05))
savefig(fig,"F19_final_AP_F1_MCC_comparison.png",caption)
fig,ax=plt.subplots(figsize=(7,5))
for mid,col in zip(IDS,COLORS):
    p,r,_=precision_recall_curve(y,scores[mid]); ax.step(r,p,where="post",color=col,label=f"{mid}: AP={metrics.loc[mid,'AP']:.3f}")
ax.axhline(TEST_PREVALENCE,color="gray",ls="--",label="Observed prevalence")
ax.set(xlabel="Recall",ylabel="Precision",title="Precision–recall curves",xlim=(0,1),ylim=(0,1.02)); ax.legend(fontsize=9)
savefig(fig,"F20_final_PR_curves.png",caption)
fig,ax=plt.subplots(figsize=(7,5))
for mid,col in zip(IDS,COLORS):
    fpr,tpr,_=roc_curve(y,scores[mid]); ax.plot(fpr,tpr,color=col,label=f"{mid}: AUC={metrics.loc[mid,'ROC_AUC']:.3f}")
ax.plot([0,1],[0,1],"--",color="gray"); ax.set(xlabel="False positive rate",ylabel="True positive rate",title="ROC curves",xlim=(0,1),ylim=(0,1.02)); ax.legend(fontsize=9)
savefig(fig,"F21_final_ROC_curves.png",caption)
fig,axes=plt.subplots(2,3,figsize=(11,7))
for mid,ax in zip(IDS,axes.flat):
    cm=confusion_matrix(y,predictions[mid],labels=[0,1]); normalized=cm/cm.sum(axis=1,keepdims=True)
    ax.imshow(normalized,vmin=0,vmax=1,cmap="Blues")
    for i in range(2):
        for j in range(2): ax.text(j,i,f"{cm[i,j]:,}\n({normalized[i,j]:.1%})",ha="center",va="center",color="white" if normalized[i,j]>.6 else "black")
    ax.set(xticks=[0,1],yticks=[0,1],xticklabels=["Benign","Phishing"],yticklabels=["Benign","Phishing"],xlabel="Predicted",ylabel="True",title=mid+" "+NAMES[mid])
axes.flat[-1].axis("off")
savefig(fig,"F22_final_confusion_matrices.png",f"{SCOPE}; raw counts with row percentages; color range fixed 0–1 across models.")
fig,ax=plt.subplots(figsize=(9,4.5))
deltas.pivot(index="metric",columns="comparison",values="hybrid_minus_url").loc[DELTA_METRICS].plot.bar(ax=ax,color=["#0072B2","#D55E00"],rot=20)
ax.axhline(0,color="black",lw=.8); ax.set(ylabel="Hybrid minus URL (metric units)",xlabel="",title="Paired metric differences; negative FPR is desirable")
savefig(fig,"F23_hybrid_minus_url_metric_delta.png",f"{SCOPE}; descriptive paired differences; AP/F1 on 0–1 scale; MCC on −1–1 scale; CIs in F25.")


In [ ]:
# CELL 7 — Exactly two predeclared paired McNemar tests, F24
mcn_rows=[]
for url,hybrid in PAIRS:
    a=predictions[url]==y; b=predictions[hybrid]==y
    both=int((a&b).sum()); uw=int((a&~b).sum()); hw=int((~a&b).sum()); neither=int((~a&~b).sum())
    contingency=np.array([[both,uw],[hw,neither]])
    require(contingency.sum()==n,"McNemar same-row count failed.")
    exact=(uw+hw)<25
    result=mcnemar(contingency,exact=exact,correction=not exact)
    direction="hybrid fewer errors" if hw>uw else "hybrid more errors" if hw<uw else "equal errors"
    mcn_rows.append({"comparison":hybrid+"-"+url,"both_correct":both,"url_correct_hybrid_wrong":uw,
        "url_wrong_hybrid_correct":hw,"both_wrong":neither,"discordant":uw+hw,"n":n,
        "statistic":float(result.statistic),"p_value":float(result.pvalue),"alpha":.05,
        "method":"exact binomial" if exact else "asymptotic continuity-corrected",
        "significant_unadjusted":bool(result.pvalue<.05),"direction":direction,
        "interpretation":direction+f"; {'significant' if result.pvalue<.05 else 'not significant'} at unadjusted alpha=.05; two predeclared comparisons"})
mcn=pd.DataFrame(mcn_rows); save_table("metrics/mcnemar_tests.csv",mcn)
fig,ax=plt.subplots(figsize=(8,4.5))
mcn.set_index("comparison")[["url_correct_hybrid_wrong","url_wrong_hybrid_correct"]].rename(columns={"url_correct_hybrid_wrong":"URL correct; hybrid wrong","url_wrong_hybrid_correct":"URL wrong; hybrid correct"}).plot.bar(ax=ax,color=["#D55E00","#009E73"],rot=0)
ax.set(xlabel="Predeclared pair",ylabel="Discordant test rows",title="McNemar discordances"); ax.legend(fontsize=9)
savefig(fig,"F24_mcnemar_discordant_predictions.png",f"{SCOPE}; same-row frozen predictions; exact test when discordance <25, otherwise corrected asymptotic.")
print(mcn[["comparison","p_value","method","interpretation"]].to_string(index=False))


In [ ]:
# CELL 8 — Paired row bootstrap, exact tie-aware weighted computation, F25
# Resample n row indices with replacement. Bincount multiplicities preserve the identical
# empirical bootstrap sample for all models. Pre-sorting scores once avoids 2,500 sorts.
def prepare_bootstrap(yy, pp, ss):
    order=np.argsort(-ss,kind="stable")
    ends=np.r_[np.flatnonzero(np.diff(ss[order]) != 0),len(ss)-1]
    return order,ends,2*yy+pp

def bootstrap_triplet(yy, pp, ss, weights, prepared):
    order,ends,codes=prepared
    tn,fp,fn,tp=np.bincount(codes,weights=weights,minlength=4)
    f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.
    denom=(tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)
    mcc=(tp*tn-fp*fn)/np.sqrt(denom) if denom else 0.
    positive=np.cumsum((weights*yy)[order])[ends]
    total=np.cumsum(weights[order])[ends]
    keep=total>0; positive=positive[keep]; total=total[keep]
    ap=np.sum(np.diff(np.r_[0.,positive])*(positive/total))/float(np.dot(weights,yy))
    return np.array([f1,mcc,ap])
prepared={mid:prepare_bootstrap(y,predictions[mid],scores[mid]) for mid in IDS}
# Verify optimized computation against sklearn on identical resampled rows, including tied scores.
rng_check=np.random.default_rng(2026); check_ix=rng_check.integers(0,n,n); check_w=np.bincount(check_ix,minlength=n)
for mid in IDS:
    got=bootstrap_triplet(y,predictions[mid],scores[mid],check_w,prepared[mid])
    expected=[f1_score(y[check_ix],predictions[mid][check_ix],zero_division=0),matthews_corrcoef(y[check_ix],predictions[mid][check_ix]),average_precision_score(y[check_ix],scores[mid][check_ix])]
    require(np.allclose(got,expected,atol=1e-12),"Weighted-bootstrap equivalence failed: "+mid)
rng_boot=np.random.default_rng(SEED)
boot=np.empty((BOOTSTRAP_B,len(IDS),3),dtype=float)
success=0; rejected=0; draws=0; boot_start=time.perf_counter()
while success<BOOTSTRAP_B:
    draws+=1
    require(draws<=BOOTSTRAP_B+1000,"Too many single-class resamples; report blocker.")
    ix=rng_boot.integers(0,n,n); weights=np.bincount(ix,minlength=n)
    positives=np.dot(weights,y)
    if positives==0 or positives==n: rejected+=1; continue
    for j,mid in enumerate(IDS): boot[success,j]=bootstrap_triplet(y,predictions[mid],scores[mid],weights,prepared[mid])
    success+=1
    if success%50==0 or success==BOOTSTRAP_B: print(f"Bootstrap {success}/{BOOTSTRAP_B}; elapsed {time.perf_counter()-boot_start:.1f}s",flush=True)
require(np.isfinite(boot).all(),"Nonfinite bootstrap metrics.")
ci_rows=[]
for j,mid in enumerate(IDS):
    for k,metric in enumerate(["F1","MCC","AP"]):
        lo,hi=np.percentile(boot[:,j,k],[2.5,97.5])
        ci_rows.append({"kind":"model","comparison":mid,"metric":metric,"estimate":metrics.loc[mid,metric],"ci_low":lo,"ci_high":hi,"successful_B":success,"seed":SEED,"method":"paired row bootstrap; 95% percentile"})
for url,hybrid in PAIRS:
    for k,metric in enumerate(["F1","MCC","AP"]):
        v=boot[:,IDS.index(hybrid),k]-boot[:,IDS.index(url),k]; lo,hi=np.percentile(v,[2.5,97.5])
        ci_rows.append({"kind":"paired_difference","comparison":hybrid+"-"+url,"metric":metric,
            "estimate":metrics.loc[hybrid,metric]-metrics.loc[url,metric],"ci_low":lo,"ci_high":hi,"successful_B":success,"seed":SEED,"method":"paired row bootstrap; 95% percentile"})
ci=pd.DataFrame(ci_rows); save_table("metrics/bootstrap_confidence_intervals.csv",ci)
BOOT_INFO={"requested_B":BOOTSTRAP_B,"successful_B":success,"one_class_rejected":rejected,"draws":draws,"seed":SEED,
    "sample_size":n,"scheme":"n iid row indices with replacement; same weights all models; fixed tied score groups",
    "interval":"95% empirical percentile [2.5,97.5]","elapsed_seconds":time.perf_counter()-boot_start,
    "limitation":"Row intervals do not account for dependence between related pages/domains."}
fig,ax=plt.subplots(figsize=(8,5)); dif=ci[ci.kind=="paired_difference"].reset_index(drop=True)
for i,row in dif.iterrows():
    col="#0072B2" if row.comparison=="M2-M1" else "#D55E00"
    ax.plot([row.ci_low,row.ci_high],[i,i],color=col,lw=2); ax.plot(row.estimate,i,"o",color=col)
ax.axvline(0,color="gray",ls="--"); ax.set(yticks=range(len(dif)),yticklabels=dif.comparison+" · "+dif.metric,xlabel="Hybrid minus URL (metric units)",title="Paired bootstrap differences with 95% percentile CIs")
savefig(fig,"F25_bootstrap_difference_CI_forest.png",f"{SCOPE}; B={success}, seed=42; identical resampled rows for all models; point = observed difference.")


In [ ]:
# CELL 9 — M2 standardized logistic coefficients, F26
lr=models["M2"].named_steps["model"]
require(lr.coef_.shape==(1,25) and lr.classes_.tolist()==[0,1],"M2 coefficient interpretation contract failed.")
coeff=pd.DataFrame({"feature":F,"family":["URL"]*15+["HTML"]*10,"coefficient":lr.coef_[0]})
coeff["abs_coefficient"]=coeff.coefficient.abs()
coeff["signed_rank_desc"]=coeff.coefficient.rank(method="min",ascending=False).astype(int)
coeff["absolute_rank_desc"]=coeff.abs_coefficient.rank(method="min",ascending=False).astype(int)
coeff["interpretation"]="phishing log-odds association per one training-standardized feature unit, conditional on other features"
coeff=coeff.sort_values("coefficient",ascending=True)
save_table("tables/lr_standardized_coefficients.csv",coeff)
fig,ax=plt.subplots(figsize=(10,9))
ax.barh(coeff.feature,coeff.coefficient,color=np.where(coeff.coefficient>=0,"#D55E00","#0072B2")); ax.axvline(0,color="gray",lw=.8)
ax.set(xlabel="Signed standardized coefficient (phishing log-odds)",title="M2 URL + HTML logistic regression coefficients")
savefig(fig,"F26_lr_standardized_coefficients.png","Frozen training-fitted imputation/scaling; coefficients reflect conditional association, not causation. Correlation affects magnitude/sign.")


In [ ]:
# CELL 10 — M4 post-hoc AP permutation importance on the predeclared subset, F27
print(f"Permutation importance: {len(perm_indices):,} fixed rows × 25 features × 5 repeats. This may take several minutes.",flush=True)
perm_start=time.perf_counter()
importance=permutation_importance(models["M4"],df.loc[perm_indices,F],y[perm_indices],scoring="average_precision",
    n_repeats=PERMUTATION_REPEATS,random_state=SEED,n_jobs=1)
# n_jobs=1 avoids concurrent forest copies; forest's frozen internal n_jobs is unchanged.
PERM_INFO["elapsed_seconds"]=time.perf_counter()-perm_start
imp=pd.DataFrame({"feature":F,"mean_AP_decrease":importance.importances_mean,"std_AP_decrease":importance.importances_std})
for repeat in range(PERMUTATION_REPEATS): imp[f"repeat_{repeat+1}"]=importance.importances[:,repeat]
imp["sample_n"]=len(perm_indices); imp["sample_prevalence"]=PERM_INFO["sample_prevalence"]
imp=imp.sort_values("mean_AP_decrease")
require(len(imp)==25 and np.isfinite(importance.importances).all(),"Invalid permutation importance.")
save_table("tables/rf_permutation_importance.csv",imp)
fig,ax=plt.subplots(figsize=(10,9))
ax.barh(imp.feature,imp.mean_AP_decrease,xerr=imp.std_AP_decrease,color="#009E73",error_kw={"elinewidth":.8})
ax.axvline(0,color="gray",lw=.8); ax.set(xlabel="Decrease in subset AP after permutation (mean ± SD)",title="M4 permutation importance — post-hoc description")
savefig(fig,"F27_rf_permutation_importance.png",f"{SCOPE}; n={len(perm_indices):,}; seed=42; 5 repeats; prevalence={PERM_INFO['sample_prevalence']:.3f}. Correlated features can share importance; SD is not a CI.")


In [ ]:
# CELL 11 — Same-machine timing and verified model file size, F28–F29
CPU_CONTEXT={"platform":platform.platform(),"processor":platform.processor(),"logical_cpus":os.cpu_count(),
    "physical_cpus":psutil.cpu_count(logical=False),"total_RAM_GiB":psutil.virtual_memory().total/2**30,
    "thread_env":{k:os.environ.get(k,"unset") for k in ["OMP_NUM_THREADS","MKL_NUM_THREADS","OPENBLAS_NUM_THREADS"]}}
try:
    from threadpoolctl import threadpool_info
    CPU_CONTEXT["threadpools"]=threadpool_info()
except ImportError: CPU_CONTEXT["threadpools"]="unavailable"
efficiency=pd.DataFrame(timing_rows)
efficiency["model_bytes"]=[MODEL_RECORDS[m]["bytes"] for m in efficiency.model_id]
efficiency["model_MiB"]=efficiency.model_bytes/2**20
efficiency["model_sha256"]=[MODEL_RECORDS[m]["sha256"] for m in efficiency.model_id]
efficiency["model_size_scope"]="frozen NB3 artifact" if RUN_MODE=="FULL" else "synthetic fixture"
efficiency["cpu_logical"]=CPU_CONTEXT["logical_cpus"]
save_table("tables/efficiency_summary.csv",efficiency)
fig,ax=plt.subplots(figsize=(8,4.5))
efficiency.set_index("model_id")[["predict_proba_seconds","predict_seconds"]].plot.bar(ax=ax,stacked=True,color=["#0072B2","#009E73"],rot=0)
ax.set(xlabel="Frozen model",ylabel="Seconds on all evaluation rows",title="Same-machine inference runtime"); ax.legend(["predict_proba","predict"],fontsize=9)
savefig(fig,"F28_inference_runtime_comparison.png",f"{SCOPE}; n={n:,}; one timed call/API after 128-row warmup; {os.cpu_count()} logical CPUs. Machine-specific, not a repeated benchmark.")
fig,ax=plt.subplots(figsize=(8,4.5)); ax.bar(efficiency.model_id,efficiency.model_MiB,color=COLORS)
ax.set(xlabel="Frozen model",ylabel="Serialized file size (MiB; 2²⁰ bytes)",title="Model artifact sizes")
savefig(fig,"F29_model_file_size_comparison.png",f"{SCOPE}; actual verified file bytes. Compressed artifact size is not inference RAM or peak memory.")


In [ ]:
# CELL 12 — Honest published-context fallback, F30
# No verified external CSV was supplied for this implementation. Do not auto-consume
# later files or infer comparability from an unverified score. Preserve this fallback.
context=metrics[["model_id","model","AP","prevalence","n"]].reset_index(drop=True).merge(pr90[["model_id","precision_at_recall","achieved_recall"]],on="model_id",validate="one_to_one")
context["published_reference"]="not supplied / not verified"
context["comparability"]="same-test published AP and P@R=.90 comparator unavailable"
context["data_version"]="phreshphish 1.0.1" if RUN_MODE=="FULL" else "fabricated synthetic data"
save_table("tables/phreshphish_context_comparison.csv",context)
fig,ax=plt.subplots(figsize=(8,5)); ax.bar(context.model_id,context.AP,color=COLORS)
ax.axhline(TEST_PREVALENCE,color="gray",ls="--",label="Observed prevalence")
ax.set(xlabel="This project's frozen models",ylabel="Average Precision",ylim=(0,1.05),title="This project's official-test AP" if RUN_MODE=="FULL" else "Synthetic AP — pipeline check only")
ax.text(.5,.96,"Same-test published AP comparator unavailable",transform=ax.transAxes,ha="center",va="top",fontsize=10,bbox=dict(facecolor="white",edgecolor="#cccccc",pad=5))
ax.legend(loc="lower right",fontsize=9)
savefig(fig,"F30_phreshphish_test_AP_context.png","No external benchmark bars: published values/protocol comparability have not been verified. No 5% base-rate experiment.")


In [ ]:
# CELL 13 — Thesis-ready summary tables, figure/table registries
summary=metrics.reset_index(drop=True).merge(pr90,on="model_id",validate="one_to_one").merge(
    efficiency[["model_id","combined_seconds","combined_ms_per_1000","model_MiB"]],on="model_id",validate="one_to_one")
for metric in ["F1","MCC","AP"]:
    subset=ci[(ci.kind=="model")&(ci.metric==metric)][["comparison","ci_low","ci_high"]].rename(columns={"comparison":"model_id","ci_low":metric+"_ci_low","ci_high":metric+"_ci_high"})
    summary=summary.merge(subset,on="model_id",validate="one_to_one")
summary["predictor_count"]=[len(FEATURES[m]) if m!="M0" else 0 for m in summary.model_id]
summary["frozen_threshold"]=.5
save_table("tables/final_model_summary.csv",summary)
feature_table=pd.DataFrame({"order":range(1,26),"feature":F,"family":["URL"]*15+["HTML"]*10})
save_table("tables/frozen_predictors.csv",feature_table)
figure_registry=pd.DataFrame(figure_rows)
require(len(figure_registry)==12 and figure_registry.figure.tolist()==[f"F{i}" for i in range(19,31)],"F19–F30 inventory failed.")
figure_registry.to_csv(OUT/"reports/figure_register.csv",index=False)
table_registry=pd.DataFrame([{"file":rel,"rows":len(frame),"columns":len(frame.columns),"scope":SCOPE} for rel,frame in tables.items()])
table_registry.to_csv(OUT/"reports/table_register.csv",index=False)
print("Prepared",len(tables),"tables and all 12 figure records.")


In [ ]:
# CELL 14 — Aligned predictions, workbook and result report; no full URLs exported
prediction_frame=pd.DataFrame({"sha256":df.sha256.to_numpy(),"label":y,"date":dates.to_numpy()})
for mid in IDS:
    prediction_frame[mid+"_prediction"]=predictions[mid]
    prediction_frame[mid+"_score"]=scores[mid]
require(len(prediction_frame)==n and prediction_frame.sha256.is_unique,"Prediction identity mismatch.")
require(hashlib.sha256("\n".join(prediction_frame.sha256).encode()).hexdigest()==ROW_HASH,"Prediction row alignment changed.")
prediction_path=OUT/"predictions/final_test_predictions.parquet"
prediction_frame.to_parquet(prediction_path,index=False)
# Round-trip comparison catches persistence or column-order mistakes.
prediction_reloaded=pd.read_parquet(prediction_path)
pd.testing.assert_frame_equal(prediction_frame,prediction_reloaded,check_dtype=False)
del prediction_reloaded
provenance_table=pd.DataFrame([{"key":"scope","value":SCOPE},{"key":"run_utc","value":RUN_ID},
    {"key":"dataset","value":json.dumps(DATASET)},{"key":"boundary_caveat","value":BOUNDARY},
    {"key":"frozen_config_sha256","value":CONFIG_HASH},{"key":"bootstrap","value":json.dumps(BOOT_INFO)},
    {"key":"permutation","value":json.dumps(safe_json(PERM_INFO))},{"key":"cpu_context","value":json.dumps(CPU_CONTEXT)},
    {"key":"packages","value":json.dumps(packages)},{"key":"repeat_exposure_note","value":REPEAT_EXPOSURE_NOTE}])
workbook=OUT/"tables/NB04_Final_Results_Master.xlsx"
with pd.ExcelWriter(workbook,engine="openpyxl") as writer:
    sheets={"Final summary":summary,"Metrics":metrics.reset_index(drop=True),"Confusion":tables["metrics/final_confusion_counts.csv"],
        "Precision at recall":pr90,"McNemar":mcn,"Bootstrap CIs":ci,"Hybrid deltas":deltas,"M2 coefficients":coeff,
        "M4 permutation":imp,"Efficiency":efficiency,"Published context":context,"Predictors":feature_table,
        "Provenance":provenance_table,"Figures":figure_registry,"Tables":table_registry}
    for name,frame in sheets.items():
        frame.to_excel(writer,sheet_name=name,index=False)
        ws=writer.sheets[name]; ws.freeze_panes="A2"; ws.auto_filter.ref=ws.dimensions
        for col in ws.columns:
            letter=col[0].column_letter; width=min(60,max(12,max(len(str(c.value or "")) for c in col)+2)); ws.column_dimensions[letter].width=width
            for cell in col[1:]:
                if isinstance(cell.value,float): cell.number_format="0.000000"
        for cell in ws[1]:
            cell.font=openpyxl.styles.Font(bold=True,color="FFFFFF"); cell.fill=openpyxl.styles.PatternFill("solid",fgColor="23445A")
wb=openpyxl.load_workbook(workbook,read_only=True)
require(set(wb.sheetnames)==set(sheets),"Workbook sheet verification failed."); wb.close()
lines=["# Notebook 4 — Final result values for thesis", "", f"**{SCOPE}**", "",
    f"Run UTC: {RUN_ID}. Evaluated rows: {n:,}; benign: {int((y==0).sum()):,}; phishing: {int(y.sum()):,}; observed prevalence: {TEST_PREVALENCE:.6f}.",
    "",BOUNDARY,"", "All classifications use frozen estimator.predict(); no refit, tuning, calibration or threshold change.",
    "AP is sklearn average_precision_score. Precision, recall and F1 use phishing=1; zero-division classification metrics are zero.",
    "", "## Frozen-model results", "", "| Model | AP [95% CI] | F1 [95% CI] | MCC [95% CI] | FPR | P@R≥.90 (achieved recall) |", "|---|---|---|---|---|---|"]
for _,r in summary.iterrows():
    lines.append(f"| {r.model_id}: {r.model} | {r.AP:.6f} [{r.AP_ci_low:.6f}, {r.AP_ci_high:.6f}] | {r.F1:.6f} [{r.F1_ci_low:.6f}, {r.F1_ci_high:.6f}] | {r.MCC:.6f} [{r.MCC_ci_low:.6f}, {r.MCC_ci_high:.6f}] | {r.FPR:.6f} | {r.precision_at_recall:.6f} ({r.achieved_recall:.6f}) |")
lines += ["", "M0 predicts benign for every row: AP equals prevalence and ROC-AUC is 0.5. Its zero class-1 score is expected.",
    "P@R=.90 uses the highest observed threshold attaining target recall, keeping all tied scores. This descriptive operating point is NOT a deployment threshold.",
    "", "## Paired differences and inference", ""]
for _,r in ci[ci.kind=="paired_difference"].iterrows(): lines.append(f"- {r.comparison}, {r.metric}: difference {r.estimate:+.6f}, 95% paired percentile CI [{r.ci_low:+.6f}, {r.ci_high:+.6f}].")
for _,r in mcn.iterrows(): lines.append(f"- McNemar {r.comparison}: URL correct/hybrid wrong={r.url_correct_hybrid_wrong:,}; URL wrong/hybrid correct={r.url_wrong_hybrid_correct:,}; p={r.p_value:.6g}; {r.method}; {r.interpretation}.")
for _,r in deltas[deltas.metric=="FPR"].iterrows(): lines.append(f"- {r.comparison} FPR difference: {r.percentage_point_difference:+.4f} percentage points; negative is desirable.")
lines += ["",f"Bootstrap: {success} successful paired samples of n rows with replacement, seed 42, {rejected} one-class samples rejected. Each row's multiplicity is shared across all models. CIs are not p-values; AP improvement alone does not establish McNemar significance.",
    "Two predeclared McNemar tests use unadjusted alpha=.05 (no familywise multiplicity adjustment). Row bootstrap assumes independent rows; related-domain dependence may narrow intervals. Test prevalence differs from development validation prevalence 28434/100775 ≈ 0.282153; AP depends on prevalence. No rebalancing applied.",
    "", "## Interpretation and efficiency", "",
    "M2 coefficients are signed effects on phishing log-odds per training-standardized feature unit, conditional on other included predictors; correlations affect magnitude/sign. They are associations, not causal effects. Saved imputation/scaling remain frozen.",
    f"M4 permutation importance used {len(perm_indices):,} uniform random test rows without replacement (seed 42, fixed before scores), sample prevalence={PERM_INFO['sample_prevalence']:.6f}, AP scoring, five repeats. Mean AP drop and SD are subset-dependent descriptive values; correlated predictors may share importance. No feature reselection follows.",
    "Efficiency includes saved artifact sizes plus one timed full predict_proba and predict call after consistent 128-row warmup. Latency is normalized to 1,000 rows. Frame construction/export is excluded. Values depend on hardware/threading; RSS snapshots are not peak RAM. NB4 does not measure training time.",
    "", "## Published comparison and limitations", "",
    "Same-test published AP and P@R=.90 values were not supplied or verified. Context CSV marks them unavailable and F30 plots only this project's measured results. No direct superiority to an external published baseline is claimed. Optional 5% base-rate benchmark is off.",
    "The official test is a one-time holdout; do not select new hyperparameters, features or thresholds after seeing these results. Any correction causing repeated exposure must be documented.",
    "Repeated-exposure note: "+(REPEAT_EXPOSURE_NOTE or "none declared"), "", "Completion is established only by Cell 15's manifest and full_complete flag."]
if RUN_MODE=="SMOKE": lines.insert(4,"**Synthetic fabricated data and models only. None of these numbers are final scientific findings.**")
(OUT/"reports/final_result_values_for_thesis.md").write_text("\n".join(lines)+"\n",encoding="utf-8")
print("Predictions, compact workbook and thesis-value report exported.")


In [ ]:
# CELL 15 — Integrity gates, atomic final manifest and optional self-excluding ZIP
required=["predictions/final_test_predictions.parquet","metrics/final_test_metrics.csv","metrics/final_confusion_counts.csv",
 "metrics/mcnemar_tests.csv","metrics/bootstrap_confidence_intervals.csv","metrics/hybrid_minus_url_differences.csv",
 "metrics/precision_at_recall_090.csv","tables/NB04_Final_Results_Master.xlsx","tables/lr_standardized_coefficients.csv",
 "tables/rf_permutation_importance.csv","tables/efficiency_summary.csv","tables/phreshphish_context_comparison.csv",
 "tables/final_model_summary.csv","reports/final_result_values_for_thesis.md","reports/figure_register.csv","reports/table_register.csv"]
required=sorted(set(required+list(tables)+figures+([str(Path(f).with_suffix(".pdf")) for f in figures] if SAVE_PDF else [])))
for rel in required: require((OUT/rel).is_file() and (OUT/rel).stat().st_size>0,"Missing/empty output: "+rel)
require(len(figures)==12 and len(set(figures))==12,"Need exactly 12 required figures.")
require(len(metrics)==5 and len(ci)==21 and len(mcn)==2 and set(mcn.comparison)=={"M2-M1","M4-M3"},"Metric/statistic inventory failed.")
require(len(coeff)==25 and len(imp)==25 and PERMUTATION_REPEATS>=5,"Interpretation incomplete.")
require(success>= (500 if RUN_MODE=="FULL" else 20),"Insufficient successful bootstrap resamples.")
require(np.isfinite(ci[["estimate","ci_low","ci_high"]].to_numpy()).all() and (ci.ci_low<=ci.ci_high).all(),"Invalid confidence intervals.")
for mid,model in models.items():
    require(joblib.hash(model)==MODEL_STATE_HASHES[mid],mid+" model changed during evaluation/interpretation.")
    require(sha256_file(model_paths[mid])==MODEL_RECORDS[mid]["sha256"],mid+" model file changed.")
for name,path in inputs.items(): require(sha256_file(path)==input_records[name]["sha256"],"Input changed: "+name)
require(hashlib.sha256("\n".join(prediction_frame.sha256).encode()).hexdigest()==ROW_HASH and len(prediction_frame)==n,"Final row identity mismatch.")
output_records={rel:file_record(OUT/rel) for rel in required}
full_complete=RUN_MODE=="FULL" and OFFICIAL_EVALUATED and not blockers
smoke_complete=RUN_MODE=="SMOKE" and not blockers
manifest={**status,"notebook":"04_final_temporal_evaluation_and_interpretation.ipynb","physical_cells":15,
 "packages":packages,"dataset":DATASET if RUN_MODE=="FULL" else {"id":"synthetic fabricated fixtures"},
 "inputs":input_records,"artifacts":output_records,"frozen_config_sha256":CONFIG_HASH,"frozen_configuration":config,
 "model_files":MODEL_RECORDS,"model_unchanged":True,"feature_order":FEATURES,"row_order_sha256":ROW_HASH,
 "test_counts":{"n":n,"benign":int((y==0).sum()),"phishing":int(y.sum()),"prevalence":TEST_PREVALENCE},
 "dates":{"min":dates.min().isoformat(),"max":dates.max().isoformat()},"metrics":metric_rows,"precision_at_recall":pr90_rows,
 "hybrid_differences":deltas.to_dict("records"),"mcnemar":mcn_rows,"bootstrap":BOOT_INFO,"bootstrap_CIs":ci_rows,
 "permutation":PERM_INFO,"timing":efficiency.to_dict("records"),"cpu_context":CPU_CONTEXT,
 "boundary_caveat":BOUNDARY,"published_comparator":"not supplied / not verified","five_percent_benchmark":False,
 "figures":figures,"no_real_model_refit_or_retuning":True,"threshold":.5,"prediction_rule":"saved predict(); class-1 probability for AP/AUC",
 "official_test_loaded":RUN_MODE=="FULL","official_test_evaluated":OFFICIAL_EVALUATED,
 "full_complete":full_complete,"smoke_complete":smoke_complete,"blockers":blockers,
 "elapsed_seconds_before_zip":time.perf_counter()-STARTED,
 "hash_policy":"Manifest excludes itself; ZIP external and excludes itself; synthetic fixtures/ZIP extraction are not result artifacts."}
# Recheck before committing success; failures leave the incomplete status manifest in place.
for rel,rec in output_records.items(): require(sha256_file(OUT/rel)==rec["sha256"],"Output changed during integrity check: "+rel)
write_json(MANIFEST_PATH,manifest)
readback=json.loads(MANIFEST_PATH.read_text()); require(readback["full_complete"]==full_complete,"Manifest readback failed.")
if MAKE_ZIP:
    zip_path=OUT.parent/f"NB04_{RUN_MODE}_ALL_OUTPUTS.zip"
    members=required+["manifests/manifest_nb04.json"]
    total=sum((OUT/r).stat().st_size for r in members)
    if shutil.disk_usage(OUT).free < total*1.1+256*2**20:
        print("Optional ZIP skipped: insufficient disk. Preserve individual verified outputs.")
    else:
        tmp=zip_path.with_suffix(".zip.tmp")
        with zipfile.ZipFile(tmp,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=1,allowZip64=True) as z:
            for rel in members: z.write(OUT/rel,arcname=rel)
        with zipfile.ZipFile(tmp) as z: require(z.testzip() is None and set(z.namelist())==set(members),"ZIP CRC/inventory failed.")
        tmp.replace(zip_path); print("ZIP:",zip_path,"bytes:",zip_path.stat().st_size)
print("Mode:",RUN_MODE,"\nOfficial test evaluated:",OFFICIAL_EVALUATED,"\nBlockers:",blockers)
print("full_complete:",full_complete,"\nsmoke_complete:",smoke_complete)
print("FULL PASS — save the executed version with outputs; do not retune." if full_complete else "SMOKE PASS — synthetic only; full_complete=False. Attach FULL inputs, restart and run FULL once.")
print("Results:",OUT)
